In [7]:
# 只需安装这 3 个（100%成功）
# pip install pandas scikit-learn requests

import pandas as pd
import json
import requests

# ===================== 配置（你只改这里！） =====================
CSV_PATH = "数据参考.csv"      # 你的文件路径
CONTENT_COL = "content"            # 新闻正文列名
API_KEY = "sk-12c7e71f8c0c4ecf9a4e7f2a5a62d743"       # 你的密钥
SAVE_PATH = "最后一次.csv"
# ===============================================================

# 1. 读取前20条数据（你要50条就改成 head(50)）
df = pd.read_csv(CSV_PATH).dropna(subset=[CONTENT_COL])
df = df.reset_index(drop=True)
print("读取数据：", len(df), "条")

# ===================== 你自己写的规则 =====================
RULE = """
你是资深反诈专家,严格按规则标注,禁止自由发挥,只输出JSON。
【诈骗标注规则库 - 严格强制执行】
=================================================

1. 【诈骗行为信息提取】(无论文本类型,只要描述了诈骗行为,就提取以下4个维度;无诈骗行为则留空)
   (1)诈骗类型(单选):
      刷单返利类
      虚假网络投资理财类
      虚假购物服务类
      冒充电商物流客服类
      虚假贷款类
      虚假征信类
      冒充领导熟人类
      冒充公检法及政府机关类
      网络婚恋、交友类
      网络游戏产品虚假交易类
      其他/新型诈骗:无法归入上述类别的都归入这个。

   (2)接触渠道(单选):微信、QQ、抖音/快手、短信、电话、网络平台、陌生链接、招聘平台、其他
  （3）损失金额(loss_amount)：提取文本内容中受害者转账/损失金额


=================================================
【受害者画像分析(所有文本必须填写)】
1. 年龄判定(age_group):
   - 根据文中提到的年龄、身份(如"大学生"、"退休")进行推断。
   - 直接输出年龄段名称:青少年(13-17)、青年(18-35)、中年(36-59)、老年(60+)、未知。
   - 不要输出具体数字,只输出年龄段名称。
2.性别(gender)：
   -提取文本受害者描述是（男性/女性，先生/女士）如果文本里没有描述空着
3.教育水平（education_level）: 仅限 ["小学及以下", "初中", "高中/中专", "大专", "本科", "硕士及以上", "未知"]
4. 职业标准化提取(occupation):
   - 原则:映射到【标准职业标签库】。
   - 【标准职业标签库】:
     - 学生
     - 教师
     - 医护人员
     - 公务员/事业单位人员
     - 企业职员
     - 个体户/经商
     - 务工人员(含工人、服务员、快递员、司机、保洁)
     - 退休人员(含退休老干部)
     - 家庭主妇/主夫
     - 待业/无业
     - 其他(无法归类的自由职业等)
5. 经济状况(economy):
   只能输出:低收入、中等收入、高收入、贫困、富有、未知

=================================================
【行为特征分析(所有文本必须填写)】
1. 资金行为(fund_behavior):小额试水、大额投入、多次转账、犹豫不决、快速响应
2. 决策特征(decision_trait):冲动决定、多方咨询、谨慎核实、盲目相信、情绪化决策
3. 事后行为(post_behavior):立即报警、尝试挽回、隐瞒不报、反思总结、告知亲友
"""


# ===================== LLM抽取 =====================
def extract(text):
    url = "https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    prompt = f"""

【标注规则】
{RULE}

【新闻文本】
{text}

强制要求：所有能提取的字段必须全部填写，禁止空值！禁止不填写！
【输出JSON字段（严格按以下格式）】
{{
    "fraud_type": "",
    "contact_channel": "",
    "Loss_amount": "",
    "age_group": "",
    "gender": "",
    "education_level": "",
    "occupation": "",
    "economy": "",
    "fund_behavior": "",
    "decision_trait": "",
    "post_behavior": ""
}}
"""
    data = {
        "model": "qwen-plus",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.0,
        "max_tokens": 1024
    }

    try:
        response = requests.post(url, headers=headers, json=data, timeout=30)
        res_json = response.json()
        result = res_json["choices"][0]["message"]["content"]

        result = result.replace("```json", "").replace("```", "").strip()

        data_dict = json.loads(result)

        real_police_words = ["记者","通讯员","警方","派出所","民警","公安局","锦旗","破案","抓获"]
        fake_police_words = [
            "冒充警察","冒充民警","冒充公检法","冒充警官","假冒警察",
            "谎称警察","谎称民警","谎称是公安局","伪造警官证","假警察"
        ]

        is_real_report = any(word in text for word in real_police_words)
        is_real_fake = any(word in text for word in fake_police_words)

        if is_real_report:
            data_dict["text_type"] = "非诈骗文本-案件报道"
            if not is_real_fake and data_dict.get("fraud_type") == "冒充公检法类":
                data_dict["fraud_type"] = ""

        result = json.dumps(data_dict, ensure_ascii=False)

        print("最终修正结果:", result[:100])
        return result
    except Exception as e:
        print("API错误:", e)
        return '''{
            "fraud_type":"","scenario_reason":"",
            "contact_channel":"","induction_scripts":"","new_category_name":"",
            "age_group":"","occupation":"","economy":"",
            "fund_behavior":"","decision_trait":"","post_behavior":""
        }'''
# ===================== 批量抽取 =====================
results = []
for i, text in enumerate(df[CONTENT_COL]):
    print(f"处理第 {i+1}/{len(df)} 条")
    result = extract(text)
    try:
        results.append(json.loads(result))
    except:
        results.append({})

# 保存结果
output = pd.concat([df, pd.DataFrame(results)], axis=1)
output.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")

print("\n✅ 完成！纯LLM抽取结果已保存：", SAVE_PATH)

读取数据： 531 条
处理第 1/531 条
最终修正结果: {"fraud_type": "虚假购物服务类", "contact_channel": "网络平台", "Loss_amount": "", "age_group": "未知", "gender":
处理第 2/531 条
最终修正结果: {"fraud_type": "其他/新型诈骗", "contact_channel": "其他", "Loss_amount": "4300000", "age_group": "老年", "gen
处理第 3/531 条
最终修正结果: {"fraud_type": "其他/新型诈骗", "contact_channel": "未知", "Loss_amount": "2800000", "age_group": "中年", "gen
处理第 4/531 条
最终修正结果: {"fraud_type": "虚假购物服务类", "contact_channel": "其他", "Loss_amount": "", "age_group": "老年", "gender": "
处理第 5/531 条
最终修正结果: {"fraud_type": "其他/新型诈骗", "contact_channel": "微信", "Loss_amount": "", "age_group": "青年", "gender": "
处理第 6/531 条
最终修正结果: {"fraud_type": "虚假购物服务类", "contact_channel": "电话、网络平台", "Loss_amount": "", "age_group": "青年", "gende
处理第 7/531 条
最终修正结果: {"fraud_type": "网络游戏产品虚假交易类", "contact_channel": "网络平台", "Loss_amount": "40000", "age_group": "青年", 
处理第 8/531 条
最终修正结果: {"fraud_type": "虚假购物服务类", "contact_channel": "网络平台", "Loss_amount": "1130000", "age_group": "青年", "g
处理第 9/531 条
最终修正结果: 